# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umarfarukh786/FlyRank-task1/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

The starter slice is a cross-sectional snapshot with heavy-tailed traffic measures. I will inspect raw quantiles and log-transformed visibility before testing signals. The audit uses the observed decline proxy only as an outcome for grouped comparisons; `trend_direction` and `trend_pct` are not treated as features.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
RAW_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Starter dataset not found: {RAW_PATH}")

df = pd.read_csv(RAW_PATH)
df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)

traffic_columns = [
    "impressions_90d", "clicks_90d", "sessions_90d", "content_age_days",
    "days_since_last_update", "word_count", "avg_position",
]
distribution = df[traffic_columns].apply(pd.to_numeric, errors="coerce").describe(
    percentiles=[0.25, 0.5, 0.75, 0.95]
).T
print("Raw distributions:")
display(distribution[["count", "mean", "50%", "95%", "max"]].round(2))

log_distribution = np.log1p(df[["impressions_90d", "clicks_90d", "sessions_90d"]].clip(lower=0)).describe(
    percentiles=[0.25, 0.5, 0.75, 0.95]
).T
print("Log1p traffic distributions used for heavy-tail inspection:")
display(log_distribution[["count", "mean", "50%", "95%", "max"]].round(2))

print(f"Overall observed decline rate: {df['is_declining_label'].mean():.3f}")
print(f"Rows with avg_position == 0 (no position data): {(df['avg_position'] == 0).mean():.3f}")

Raw distributions:


,count,mean,50%,95%,max
impressions_90d,30000.0,5200.37,731.0,22996.50,517715.0
clicks_90d,30000.0,16.10,1.0,69.05,4178.0
sessions_90d,30000.0,37.07,7.0,166.00,4345.0
content_age_days,30000.0,256.17,236.0,487.00,564.0
days_since_last_update,30000.0,46.10,20.0,104.00,373.0
word_count,22301.0,3107.76,2877.0,6173.00,9546.0
avg_position,30000.0,16.34,10.8,48.20,245.0


Log1p traffic distributions used for heavy-tail inspection:


,count,mean,50%,95%,max
impressions_90d,30000.0,6.19,6.60,10.04,13.16
clicks_90d,30000.0,1.21,0.69,4.25,8.34
sessions_90d,30000.0,2.41,2.08,5.12,8.38


Overall observed decline rate: 0.542
Rows with avg_position == 0 (no position data): 0.040


## 2. Signal tests #1 / #2 / #3

Each test asks whether the observed decline rate varies across four equally sized signal buckets. The table always includes `n`. A bucket with fewer than 50 rows cannot support a verdict. A range in rates supports a directional audit finding, while a non-monotonic pattern is reported as MIXED rather than forced into a simple story.

The three claims are:

1. **Visibility claim:** decline rates differ across trailing-90-day impression quartiles.
2. **Freshness claim:** decline rates differ across content-age quartiles.
3. **Depth claim:** decline rates differ across measured word-count quartiles.

These are association checks, not causal tests.

In [2]:
def run_signal_test(data, column, claim):
    working = data[[column, "is_declining_label"]].copy()
    working[column] = pd.to_numeric(working[column], errors="coerce")
    working = working.dropna()
    working = working[working[column] >= 0]
    working["bucket"] = pd.qcut(working[column], q=4, duplicates="drop")
    summary = (
        working.groupby("bucket", observed=True)
        .agg(
            n=("is_declining_label", "size"),
            signal_median=(column, "median"),
            decline_rate=("is_declining_label", "mean"),
        )
        .reset_index()
    )
    rates = summary["decline_rate"].to_numpy()
    sufficient = (summary["n"] >= 50).all()
    rate_range = float(rates.max() - rates.min()) if len(rates) else 0.0
    differences = np.diff(rates)
    monotonic = bool(
        len(differences) == 0
        or np.all(differences >= -0.02)
        or np.all(differences <= 0.02)
    )
    if not sufficient:
        verdict = "INSUFFICIENT DATA"
    elif rate_range < 0.03:
        verdict = "FALSE"
    elif monotonic:
        verdict = "CONFIRMED"
    else:
        verdict = "MIXED"
    print(f"Claim: {claim}")
    display(summary.round({"signal_median": 2, "decline_rate": 3}))
    print(f"Verdict: {verdict} | rate range: {rate_range:.3f} | all bucket n >= 50: {sufficient}")
    return {"column": column, "verdict": verdict, "summary": summary, "rate_range": rate_range}

signal_results = {
    "visibility": run_signal_test(
        df,
        "impressions_90d",
        "Decline rates differ across trailing-90-day impression quartiles.",
    ),
    "freshness": run_signal_test(
        df,
        "content_age_days",
        "Decline rates differ across content-age quartiles.",
    ),
    "depth": run_signal_test(
        df,
        "word_count",
        "Decline rates differ across measured word-count quartiles.",
    ),
}

Claim: Decline rates differ across trailing-90-day impression quartiles.


,bucket,n,signal_median,decline_rate
0,"(0.999, 81.0]",7503,10.0,0.376
1,"(81.0, 731.0]",7499,300.0,0.605
2,"(731.0, 3615.25]",7498,1616.0,0.626
3,"(3615.25, 517715.0]",7500,9579.5,0.562


Verdict: MIXED | rate range: 0.250 | all bucket n >= 50: True
Claim: Decline rates differ across content-age quartiles.


,bucket,n,signal_median,decline_rate
0,"(89.999, 132.0]",7518,111.0,0.600
1,"(132.0, 236.0]",8128,174.0,0.640
2,"(236.0, 333.0]",6917,302.0,0.494
3,"(333.0, 564.0]",7437,445.0,0.422


Verdict: MIXED | rate range: 0.218 | all bucket n >= 50: True
Claim: Decline rates differ across measured word-count quartiles.


,bucket,n,signal_median,decline_rate
0,"(7.999, 2413.0]",5576,1450.0,0.497
1,"(2413.0, 2877.0]",5586,2688.0,0.595
2,"(2877.0, 3666.0]",5566,3121.0,0.583
3,"(3666.0, 9546.0]",5573,4896.0,0.599


Verdict: CONFIRMED | rate range: 0.102 | all bucket n >= 50: True


## 3. The flag-linked test

The starter baseline uses a `stale_visible_page` reason when `days_since_last_update >= 180` and `impressions_90d >= 500`. The test below checks whether that combined flag has a different observed decline rate from the rest of the slice, while showing group sizes and the overall rate. It can support prioritization, but it cannot prove that updating a page will improve traffic.

In [3]:
df["stale_visible_flag"] = (
    (pd.to_numeric(df["days_since_last_update"], errors="coerce") >= 180)
    & (pd.to_numeric(df["impressions_90d"], errors="coerce") >= 500)
)
flag_summary = (
    df.groupby("stale_visible_flag", observed=False)
    .agg(
        n=("is_declining_label", "size"),
        decline_rate=("is_declining_label", "mean"),
        median_impressions=("impressions_90d", "median"),
        median_days_since_update=("days_since_last_update", "median"),
    )
    .reset_index()
)
print("Flag-linked test: stale_visible_page")
display(flag_summary.round(3))

flagged = flag_summary.loc[flag_summary["stale_visible_flag"] == True, "decline_rate"]
unflagged = flag_summary.loc[flag_summary["stale_visible_flag"] == False, "decline_rate"]
flagged_n = flag_summary.loc[flag_summary["stale_visible_flag"] == True, "n"]
if flagged.empty or unflagged.empty or flagged_n.iloc[0] < 50:
    flag_verdict = "INSUFFICIENT DATA"
else:
    difference = float(flagged.iloc[0] - unflagged.iloc[0])
    flag_verdict = "CONFIRMED" if difference >= 0.03 else "OPPOSITE" if difference <= -0.03 else "MIXED"
    print(f"Flagged minus unflagged decline-rate difference: {difference:.3f}")
print(f"Verdict: {flag_verdict}")

Flag-linked test: stale_visible_page


,stale_visible_flag,n,decline_rate,median_impressions,median_days_since_update
0,False,29983,0.542,731.0,20.0
1,True,17,0.941,4429.0,194.0


Verdict: INSUFFICIENT DATA


## 4. What this means in practice

The audit should determine which signals are useful for prioritization, not turn an observed association into an automatic edit. Traffic measures are heavy-tailed, so the analysis reports medians, quartile buckets, and visible sample sizes rather than trusting raw means. The flag-linked test is especially useful for deciding whether stale, visible pages deserve a review lane, but a reviewer still needs to verify intent, content quality, seasonality, and tracking context.

In [4]:
print("Practical interpretation")
for name, result in signal_results.items():
    print(f"- {name}: {result['verdict']} (rate range {result['rate_range']:.3f})")
print(f"- stale_visible_page flag: {flag_verdict}")
print("Use confirmed or mixed signals as review prompts, not causal refresh instructions.")
print("Keep a monitor lane for weak evidence and rerun the audit on a later or grouped slice before operational use.")

Practical interpretation
- visibility: MIXED (rate range 0.250)
- freshness: MIXED (rate range 0.218)
- depth: CONFIRMED (rate range 0.102)
- stale_visible_page flag: INSUFFICIENT DATA
Use confirmed or mixed signals as review prompts, not causal refresh instructions.
Keep a monitor lane for weak evidence and rerun the audit on a later or grouped slice before operational use.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.